In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/Spring Semester/deep learning project

/content/drive/MyDrive/Spring Semester/deep learning project


In [3]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 94.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 13.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 8.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of tifffile to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━

### PREPROCESSING

In [ ]:
import os
from pathlib import Path

# === DATASET PATHS ===
TRAIN_DICOM_DIR = Path("/content/drive/MyDrive/Spring Semester/dataset/stage_2_train_images")
TEST_DICOM_DIR = Path("/content/drive/MyDrive/Spring Semester/dataset/stage_2_test_images")
METADATA_CSV    = Path("/content/drive/MyDrive/Spring Semester/dataset/metadata_clean.csv")
LABELS_CSV      = Path("/content/drive/MyDrive/Spring Semester/dataset/stage_2_train_labels.csv")
SAMPLE_SUB_CSV  = Path("/content/drive/MyDrive/Spring Semester/dataset/stage_2_sample_submission.csv")
CLASS_INFO_CSV  = Path("/content/drive/MyDrive/Spring Semester/dataset/stage_2_detailed_class_info.csv")

# === PROJECT ROOT ===
PROJECT_ROOT = Path("/content/drive/MyDrive/Spring Semester/deep learning project")
SRC_DIR      = PROJECT_ROOT / "src"
SCRIPTS_DIR  = PROJECT_ROOT / "scripts"
OUTPUT_CKPT  = PROJECT_ROOT / "outputs" / "checkpoints"

# === EXISTING SOURCE FILES ===
DICOM_TO_PNG_PY = PROJECT_ROOT / "src" / "dl_project" / "data" / "dicom_to_png.py"
PREPROCESSING_PY = PROJECT_ROOT / "src" / "dl_project" / "data" / "preprocessing.py"
RSNA_LABELS_PY   = PROJECT_ROOT / "src" / "dl_project" / "data" / "rsna_labels.py"
SPLITS_PY        = PROJECT_ROOT / "src" / "dl_project" / "data" / "splits.py"

CONVERT_SCRIPT   = PROJECT_ROOT / "scripts" / "data" / "convert_dicom_to_png.py"
PREPARE_SCRIPT   = PROJECT_ROOT / "scripts" / "data" / "prepare_rsna_dataset.py"

# === OUTPUT DIRS FOR THIS STAGE ===
WORK_DATA_DIR    = PROJECT_ROOT / "data"
PNG_DIR          = WORK_DATA_DIR / "png_images"
CSV_DIR          = WORK_DATA_DIR / "csv"
LOG_DIR          = PROJECT_ROOT / "outputs" / "logs"

for p in [WORK_DATA_DIR, PNG_DIR, CSV_DIR, LOG_DIR, OUTPUT_CKPT]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PNG_DIR:", PNG_DIR)
print("CSV_DIR:", CSV_DIR)
print("OUTPUT_CKPT:", OUTPUT_CKPT)

PROJECT_ROOT: /content/drive/MyDrive/Spring Semester/deep learning project
PNG_DIR: /content/drive/MyDrive/Spring Semester/deep learning project/data/png_images
CSV_DIR: /content/drive/MyDrive/Spring Semester/deep learning project/data/csv
OUTPUT_CKPT: /content/drive/MyDrive/Spring Semester/deep learning project/outputs/checkpoints


In [ ]:
paths_to_check = {
    "TRAIN_DICOM_DIR": TRAIN_DICOM_DIR,
    "TEST_DICOM_DIR": TEST_DICOM_DIR,
    "METADATA_CSV": METADATA_CSV,
    "LABELS_CSV": LABELS_CSV,
    "SAMPLE_SUB_CSV": SAMPLE_SUB_CSV,
    "CLASS_INFO_CSV": CLASS_INFO_CSV,
    "PROJECT_ROOT": PROJECT_ROOT,
    "SRC_DIR": SRC_DIR,
    "DICOM_TO_PNG_PY": DICOM_TO_PNG_PY,
    "PREPROCESSING_PY": PREPROCESSING_PY,
    "RSNA_LABELS_PY": RSNA_LABELS_PY,
    "SPLITS_PY": SPLITS_PY,
    "CONVERT_SCRIPT": CONVERT_SCRIPT,
    "PREPARE_SCRIPT": PREPARE_SCRIPT,
}

for name, path in paths_to_check.items():
    print(f"{name}: exists={path.exists()} -> {path}")

TRAIN_DICOM_DIR: exists=True -> /content/drive/MyDrive/Spring Semester/dataset/stage_2_train_images
TEST_DICOM_DIR: exists=True -> /content/drive/MyDrive/Spring Semester/dataset/stage_2_test_images
METADATA_CSV: exists=True -> /content/drive/MyDrive/Spring Semester/dataset/metadata_clean.csv
LABELS_CSV: exists=True -> /content/drive/MyDrive/Spring Semester/dataset/stage_2_train_labels.csv
SAMPLE_SUB_CSV: exists=True -> /content/drive/MyDrive/Spring Semester/dataset/stage_2_sample_submission.csv
CLASS_INFO_CSV: exists=True -> /content/drive/MyDrive/Spring Semester/dataset/stage_2_detailed_class_info.csv
PROJECT_ROOT: exists=True -> /content/drive/MyDrive/Spring Semester/deep learning project
SRC_DIR: exists=False -> /content/drive/MyDrive/Spring Semester/deep learning project/src
DICOM_TO_PNG_PY: exists=False -> /content/drive/MyDrive/Spring Semester/deep learning project/src/dl_project/data/dicom_to_png.py
PREPROCESSING_PY: exists=False -> /content/drive/MyDrive/Spring Semester/deep le

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Spring Semester/deep learning project")
SRC_DIR = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(sys.path[:10])

['/content/drive/MyDrive/Spring Semester/deep learning project/src', '/content/drive/MyDrive/Spring Semester/deep learning project', '/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages']


In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/scripts/preprocess_rsna_detection_final.py"

[OK] Loaded TRAIN_LABELS_CSV: /content/drive/MyDrive/Spring Semester/dataset/stage_2_train_labels.csv | shape=(30227, 6)
[OK] Loaded CLASS_INFO_CSV: /content/drive/MyDrive/Spring Semester/dataset/stage_2_detailed_class_info.csv | shape=(30227, 2)
[OK] Loaded METADATA_CLEAN_CSV: /content/drive/MyDrive/Spring Semester/dataset/metadata_clean.csv | shape=(30000, 5)
[INFO] metadata_clean.csv available: shape=(30000, 5)
Aggregating train records: 100% 26684/26684 [3:24:21<00:00,  2.18it/s]
[OK] Loaded SAMPLE_SUBMISSION_CSV: /content/drive/MyDrive/Spring Semester/dataset/stage_2_sample_submission.csv | shape=(3000, 2)
Building test records: 100% 3000/3000 [43:30<00:00,  1.15it/s]

[DONE] RSNA detection preprocessing completed.
[INFO] Output root     : /content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data
[INFO] Images dir      : /content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images
[INFO] Train CSV       : /content/dri

### ESKİ TRAIN

In [ ]:
!pip uninstall -y albumentations albucore opencv-python opencv-python-headless
!pip install --no-cache-dir albumentations==1.4.18 albucore==0.0.17 opencv-python-headless==4.10.0.84

Found existing installation: albumentations 1.4.10
Uninstalling albumentations-1.4.10:
  Successfully uninstalled albumentations-1.4.10
Found existing installation: albucore 0.0.24
Uninstalling albucore-0.0.24:
  Successfully uninstalled albucore-0.0.24
Found existing installation: opencv-python 4.11.0.86
Uninstalling opencv-python-4.11.0.86:
  Successfully uninstalled opencv-python-4.11.0.86
Found existing installation: opencv-python-headless 4.10.0.84
Uninstalling opencv-python-headless-4.10.0.84:
  Successfully uninstalled opencv-python-headless-4.10.0.84
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.0/224.0 kB 361.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 397.0 MB/s eta 0:00:00


In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/scripts/train_rsna_detection.py"

Device: cuda
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 64 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
Train batches: 709
Val batches  : 126
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
[Val  ][Epoch 001] step 1/126[DEBUG][VAL][Epoch 001] max_score_mean=0.9088 | max_score_max=0.9820 | foreground_ratio=0.0000
[DEBUG][VAL][Epoch 001] top

In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/scripts/train_rsna_detection.py"

Device: cuda
Train batches: 709
Val batches  : 126
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
Resume checkpoint : /content/drive/MyDrive/Spring Semester/deep learning project/outputs/diffusion_guided_detr/run2/checkpoints/epoch_032.pth
Loaded epoch      : 32
Start epoch       : 33
Best metric       : 11.134288743112256
Best epoch        : 32
Epoch 33/100
[DEBUG] First batch loaded at Epoch 33
[DEBUG] Epoch 33 | train step 1/709
[DEBUG] Epoch 33 | train step 25/709
[DEBUG] Epoch 33 | train step 50/709
[DEBUG] Epoch 33 | train step 75/709
[DEBUG] Epoch 33 | train step 100/709
[DEBUG] Epoch 33 | train step 125/709
[DEBUG] Epoch 33 | train step 150/709
[DEBUG] Epoch 33 | train step 175/709
[DEBUG] Epoch 33 | train st

In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/scripts/train_rsna_detection.py"

In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/scripts/evaluate_rsna_detection.py \
    --csv_path "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/metadata/val_master.csv" \
    --checkpoint "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/diffusion_guided_detr/run3/checkpoints/best.pth" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/diffusion_guided_detr/run3/eval_results" \
    --image_size 384 \
    --num_classes 2 \
    --batch_size 16 \
    --score_thresh 0.05 \
    --image_score_thresh 0.35 \
    --enable_threshold_sweep

/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
Checkpoint loaded.
Missing keys   : []
Unexpected keys: ['criterion.empty_weight']

RSNA DETECTION EVALUATION SUMMARY

[Detection Metrics]
AP50                            : 0.010678
AP75                            : 0.000186
mAP@[0.50:0.95]                 : 0.002306
Precision @ summary threshold   : 0.020041
Recall @ summary threshold      : 0.532821
F1 @ s

In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/scripts/train_rsna_detection.py"

Device: cuda
/content/drive/MyDrive/Spring Semester/deep learning project/scripts/dataset_rsna_detection.py:225: UserWarning: Argument(s) 'value, mask_value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/content/drive/MyDrive/Spring Semester/deep learning project/scripts/dataset_rsna_detection.py:234: UserWarning: Argument(s) 'value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(
/content/drive/MyDrive/Spring Semester/deep learning project/scripts/dataset_rsna_detection.py:270: UserWarning: Argument(s) 'value, mask_value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(
Train batches: 355
Val batches  : 63
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:306: UserWarning: enable_nested_tensor is True,

In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/scripts/evaluate_rsna_detection.py" \
    --csv_path "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/metadata/val_master.csv" \
    --checkpoint "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/diffusion_guided_detr/run3/checkpoints/best.pth" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/diffusion_guided_detr/run3/evaluation_results" \
    --image_size 384 \
    --num_classes 2 \
    --backbone_name "swin_tiny_patch4_window7_224" \
    --num_queries 100 \
    --fusion_mode "hybrid" \
    --batch_size 16 \
    --score_thresh 0.01 \
    --summary_score_thresh 0.35 \
    --image_score_thresh 0.35 \
    --image_min_positive_boxes 1 \
    --enable_threshold_sweep

/content/drive/MyDrive/Spring Semester/deep learning project/scripts/evaluate_rsna_detection.py:261: UserWarning: Argument(s) 'value, mask_value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
Checkpoint loaded.
Missing keys   : []
Unexpected keys: ['criterion.empty_weight']
Starting Evaluation on 4003 images...
[EVAL] Progress: 251/251 batches processed...

Calculation summary metrics...

RSNA DETECTION EVALUATION SUMMARY

[Detection Metrics]
AP50                            : 0.043842
AP75                            : 0.000264
mAP@[0.50:0.95]                 : 0.007782
Precision @ summary threshold   : 0.000000
Recall @ summary threshold      : 0.000000
F1 @ s

In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/scripts/predict_rsna_detection.py" \
    --input_path "/path/to/test_images_folder" \
    --checkpoint "/path/to/run3/checkpoints/best.pth" \
    --output_dir "/path/to/predictions_output" \
    --num_classes 2 \
    --score_thresh 0.35 \
    --save_visualizations

### YENİ TRAIN

In [ ]:
!pip install -U albumentations albucore

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 369.4/369.4 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 110.7 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.18.4
    Uninstalling pydantic_core-2.18.4:
      Successfully uninstalled pydantic_core-2.18.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.7.4
    Uninstalling pydantic-2.7.4:
      Successfully uninstalled pydantic-2.7.4
  Attempting uninstall: albumentations
    Found existing installation: albumentations 1.4.10
    Uninstalling albumentations-1.4.10:
      Successfully uninstalled albumentations-1.4.10
ERROR: pip's dependency resolver does not currently take into account

In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/train.py" \
    --img_dir "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images" \
    --train_csv "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/metadata/train_master.csv" \
    --val_csv "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/metadata/val_master.csv" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run4" \
    --epochs 100 \
    --batch_size 32 \
    --lr 1.5e-4 \
    --num_workers 4 \
    --alpha 0.5

🚀  Device: cuda
✅ 22681 adet kayıt başarıyla işlendi.
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
✅ 4003 adet kayıt başarıyla işlendi.
📦  Train: 22681 | Val: 4003
Epoch 01/1  train=1.0524  val=0.9708  acc=75.3%
  ⭐  New best model saved!
📊  Training curves saved to /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run4/training_curves.png

✅  Training complete. Results in /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run4


In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/evaluate.py" \
    --model_path "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run4/checkpoints/best_ayolo.pth" \
    --img_dir "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images" \
    --csv_path "/content/drive/MyDrive/Spring Semester/dataset/processed_metadata/rsna_master_metadata.csv" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run4/final_rsna_test" \
    --batch_size 128

✅  Model loaded from /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run4/checkpoints/best_ayolo.pth
📂 test split'i seçildi. Satır sayısı: 2668
✅ 2668 adet kayıt başarıyla işlendi.
📊  Evaluating 2668 samples…
100% 21/21 [12:46<00:00, 36.52s/it]

══════════════════════════════════════════════════
        🚀 A-YOLO DEĞERLENDİRME RAPORU
══════════════════════════════════════════════════
  ✨ Accuracy   : 78.71%
  🎯 Precision  : 0.5322
  🔎 Recall     : 0.4542  <-- (Hassasiyet)
  ⚖️  F1-Score   : 0.4901
  📈 AUC-ROC    : 0.7968
  📊 AP (PR)    : 0.5241
  📏 Mean IoU   : 0.0000  (Bbox başarısı)
  ✅ IoU@0.5    : 0.0%
══════════════════════════════════════════════════

🎉 Tüm sonuçlar ve grafikler şuraya kaydedildi: /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run4/final_rsna_test


In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/visualize_gradcam.py" \
    --model_path "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run4/checkpoints/best_ayolo.pth" \
    --csv_path "/content/drive/MyDrive/Spring Semester/dataset/processed_metadata/rsna_master_metadata.csv" \
    --img_dir "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run4/visual_proofs" \
    --num_samples 2

🖼️  Generating explanations for 2 samples…
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:744: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
✅  a76c9c2b-366c-47a6-97e4-c442a2f84dd2  →  SAĞLIKLI (25.7%)
✅  84b3869e-5f82-423f-9553-b1006a00a04c  →  ZATÜRRE (65.9%)

✅  Done. Results saved to /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run4/visual_proofs


In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/inference.py" \
    --model_path "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run4/checkpoints/best_ayolo.pth" \
    --img_path "/content/drive/MyDrive/Spring Semester/dataset/processed_metadata/rsna_master_metadata.csv" \
    --img_dir "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run4/inference_results" \
    --threshold 0.45 \
    --show_gradcam

📊  Processing 26684 patients from CSV…
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:744: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
🟢 NORMAL  (39.6%)  →  /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run4/inference_results/result_0004cfab-14fd-4e49-80ba-63a80b6bddd6.png
🟢 NORMAL  (21.9%)  →  /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run4/inference_results/result_00313ee0-9eaa-42f4-b0ab-c148ed3241cd.png
🟢 NORMAL  (17.3%)  →  /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run4/inference_results/result_00322d4d-1c29-4943-afc9-b6754be640eb.png
🟢 NORMAL  (7.3%)  →  /content/drive/MyDrive/Spring Semester/deep learning project/output

### 100 EPOCH

In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/train.py" \
    --img_dir "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images" \
    --train_csv "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/metadata/train_master.csv" \
    --val_csv "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/metadata/val_master.csv" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final" \
    --epochs 100 \
    --batch_size 32 \
    --lr 1.5e-4 \
    --weight_decay 0.05 \
    --alpha 0.3 \
    --patience 30 \
    --num_workers 4

🚀  Device: cuda
✅ 22681 adet kayıt başarıyla işlendi.
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
✅ 4003 adet kayıt başarıyla işlendi.
📦  Train: 22681 | Val: 4003
model.safetensors: 100% 346M/346M [00:04<00:00, 81.0MB/s]
Epoch 01 [Train]:   0% 0/709 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:143: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "
Epo

In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/evaluate.py" \
    --model_path "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final/checkpoints/best_ayolo.pth" \
    --img_dir "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images" \
    --csv_path "/content/drive/MyDrive/Spring Semester/dataset/processed_metadata/rsna_master_metadata.csv" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final/final_rsna_test" \
    --batch_size 128

✅  Model loaded from /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final/checkpoints/best_ayolo.pth
📂 test split'i seçildi. Satır sayısı: 2668
✅ 2668 adet kayıt başarıyla işlendi (Kutu sütunları: x_min, y_min kullanıldı).
📊  Evaluating 2668 samples...
  0% 0/21 [00:00<?, ?it/s]Batch içindeki gerçek hasta sayısı: 43
POZİTİF VAKA - True Box: [0.29882812 0.49902344 0.5371094  0.27734375]

Debug - Pred: [0.37914044 0.40016812 0.47704315 0.41296837] | True: [0.29882812 0.49902344 0.5371094  0.27734375] | IoU: 0.5777592658996582
POZİTİF VAKA - True Box: [0.17675781 0.1796875  0.60546875 0.55371094]

Debug - Pred: [0.2608642  0.24046543 0.56363094 0.5556932 ] | True: [0.17675781 0.1796875  0.60546875 0.55371094] | IoU: 0.656501829624176
POZİTİF VAKA - True Box: [0.35742188 0.27148438 0.5888672  0.5625    ]
POZİTİF VAKA - True Box: [0.21679688 0.25195312 0.5830078  0.45214844]
POZİTİF VAKA - True Box: [0.7265625  0.20214844 0.10058594 0.18945312]
POZİTİF VAKA 

In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/visualize_gradcam.py" \
    --model_path "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final/checkpoints/best_ayolo.pth" \
    --csv_path "/content/drive/MyDrive/Spring Semester/dataset/processed_metadata/rsna_master_metadata.csv" \
    --img_dir "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final/visual_proofs" \
    --num_samples 15

🖼️  Generating explanations for 15 samples…
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:744: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
✅  acbd5788-b2a8-4fa3-b9ff-f6738bcf6f08  →  SAĞLIKLI (8.0%)
✅  38b7377c-4b94-4afc-8738-da7da52fe760  →  ZATÜRRE (50.8%)
✅  37c522d1-93b0-4b70-9cf7-3ae932a7a70e  →  SAĞLIKLI (13.8%)
✅  779e0af8-53e5-4f43-818b-6d0bf9e1b9a5  →  ZATÜRRE (57.9%)
✅  b5ba706b-f5f7-45be-ab33-e955db947121  →  SAĞLIKLI (11.5%)
✅  e4d22644-951a-49cf-98ca-3e8119448192  →  ZATÜRRE (59.1%)
✅  0d96ad36-6fce-4ae2-a88f-faa5c7bfbf0e  →  ZATÜRRE (60.8%)
✅  5d6c55b9-000c-459a-ae15-0d3cf9d62b69  →  SAĞLIKLI (14.8%)
✅  dfc1a68e-3314-4059-8a76-4e4e0981a948  →  ZATÜRRE (88.2%)
✅  625f8968-3135-4c81-a5a5-aa61dab240c7  →  SAĞ

In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/inference.py" \
    --model_path "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final/checkpoints/best_ayolo.pth" \
    --img_path "/content/drive/MyDrive/Spring Semester/dataset/processed_metadata/rsna_master_metadata.csv" \
    --img_dir "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final/inference_results" \
    --threshold 0.45 \
    --show_gradcam

📊  Processing 26684 patients from CSV…
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:744: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
🟢 NORMAL  (21.0%)  →  /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final/inference_results/result_0004cfab-14fd-4e49-80ba-63a80b6bddd6.png
🟢 NORMAL  (26.6%)  →  /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final/inference_results/result_00313ee0-9eaa-42f4-b0ab-c148ed3241cd.png
🟢 NORMAL  (31.1%)  →  /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final/inference_results/result_00322d4d-1c29-4943-afc9-b6754be640eb.png
🟢 NORMAL  (6.6%)  →  /content/drive/MyDrive/Spring Semester/deep learning

### TRAIN 100 EPOCH

In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/evaluate.py" \
    --model_path "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final/checkpoints/best_ayolo.pth" \
    --img_dir "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images" \
    --csv_path "/content/drive/MyDrive/Spring Semester/dataset/processed_metadata/rsna_master_metadata.csv" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/final_rsna_test" \
    --batch_size 128

✅  Model loaded from /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final/checkpoints/best_ayolo.pth
📂 test split'i seçildi. Satır sayısı: 2668
📊  Evaluating 2668 samples...
100% 21/21 [00:31<00:00,  1.48s/it]

══════════════════════════════════════════════════
        🚀 A-YOLO EVALUATION REPORT
══════════════════════════════════════════════════
  ✨ Accuracy     : 76.99%
  🎯 Precision    : 0.4931
  🔎 Recall       : 0.7770  (Sensitivity)
  ⚖️  F1-Score     : 0.6034
  📈 AUC-ROC      : 0.8488
  📊 mAP (AP@50)  : 0.5187  🚀
  📏 Mean IoU     : 0.3911
  ✅ IoU >= 0.5   : 38.1%
══════════════════════════════════════════════════

🎉 All results and plots saved to: /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/final_rsna_test


In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/visualize_gradcam.py" \
    --model_path "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final/checkpoints/best_ayolo.pth" \
    --csv_path "/content/drive/MyDrive/Spring Semester/dataset/processed_metadata/rsna_master_metadata.csv" \
    --img_dir "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/visual_proofs" \
    --num_samples 15

🖼️  Generating explanations for 15 samples...
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:744: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
✅ Generated: 37e04b5c-b7ad-4074-81ee-7dad137fe063 → NORMAL (6.8%)
✅ Generated: 3be6a4c5-7c20-48cf-ae13-e18a828b3868 → PNEUMONIA (63.1%)
✅ Generated: 64b3e733-0567-4c97-b68c-3760dc45bab6 → NORMAL (2.4%)
✅ Generated: fbe58065-a841-4d40-84c3-76e932562beb → NORMAL (40.1%)
✅ Generated: 27bdb2cc-9665-40ea-a6b4-248a1112dbe8 → PNEUMONIA (58.2%)
✅ Generated: b1185bf9-79d4-4497-ac44-cd2581128f5b → PNEUMONIA (66.8%)
✅ Generated: 36592e9e-7b4e-4b39-8106-cdbf56160e07 → NORMAL (7.6%)
✅ Generated: 98325e35-8c87-418b-a7c3-81b1180e85ea → PNEUMONIA (50.4%)
✅ Generated: e3bcba66-438e-490a-aea8-3afa23

In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/inference.py" \
    --model_path "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final/checkpoints/best_ayolo.pth" \
    --img_path "/content/drive/MyDrive/Spring Semester/dataset/processed_metadata/rsna_master_metadata.csv" \
    --img_dir "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/inference_results" \
    --threshold 0.45 \
    --show_gradcam

model.safetensors: 100% 346M/346M [00:04<00:00, 76.8MB/s]
📂 Only 'test' split selected. Processing 2668 images.
📊  Processing 2668 patients from CSV...
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:744: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
🔴 PNEUMONIA  (48.0%)  →  /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/inference_results/result_00569f44-917d-4c86-a842-81832af98c30.png
🔴 PNEUMONIA  (72.4%)  →  /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/inference_results/result_00c0b293-48e7-4e16-ac76-9269ba535a62.png
🔴 PNEUMONIA  (75.0%)  →  /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/inference_resu

### TRAIN 200 EPOCH -V2

In [3]:
!pip install -U albumentations albucore

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 369.4/369.4 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 102.7 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.18.4
    Uninstalling pydantic_core-2.18.4:
      Successfully uninstalled pydantic_core-2.18.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.7.4
    Uninstalling pydantic-2.7.4:
      Successfully uninstalled pydantic-2.7.4
  Attempting uninstall: albumentations
    Found existing installation: albumentations 1.4.10
    Uninstalling albumentations-1.4.10:
      Successfully uninstalled albumentations-1.4.10
ERROR: pip's dependency resolver does not currently take into account

In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/train.py" \
    --img_dir "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images" \
    --train_csv "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/metadata/train_master.csv" \
    --val_csv "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/metadata/val_master.csv" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2" \
    --epochs 200 \
    --batch_size 32 \
    --lr 1e-4 \
    --weight_decay 0.05 \
    --alpha 0.2 \
    --patience 190 \
    --num_workers 4

🚀  Device: cuda
✅ 22681 adet kayıt başarıyla işlendi.
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
✅ 4003 adet kayıt başarıyla işlendi.
📦  Train: 22681 | Val: 4003
model.safetensors: 100% 346M/346M [00:03<00:00, 89.0MB/s]
Epoch 01 [Train]:   0% 0/709 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:143: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "
📊  

In [3]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/evaluate.py" \
    --model_path "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/checkpoints/best_ayolo.pth" \
    --img_dir "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images" \
    --csv_path "/content/drive/MyDrive/Spring Semester/dataset/processed_metadata/rsna_master_metadata.csv" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/final_rsna_test" \
    --batch_size 512



✅  Model loaded from /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/checkpoints/best_ayolo.pth
📂 test split'i seçildi. Satır sayısı: 2668
📊  Evaluating 2668 samples...
100% 6/6 [14:55<00:00, 149.18s/it]

══════════════════════════════════════════════════
        🚀 A-YOLO EVALUATION REPORT
══════════════════════════════════════════════════
  ✨ Accuracy     : 78.15%
  🎯 Precision    : 0.5103
  🔎 Recall       : 0.7404  (Sensitivity)
  ⚖️  F1-Score     : 0.6042
  📈 AUC-ROC      : 0.8578
  📊 mAP (AP@50)  : 0.5855  🚀
  📏 Mean IoU     : 0.4054
  ✅ IoU >= 0.5   : 42.6%
══════════════════════════════════════════════════

🎉 All results and plots saved to: /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/final_rsna_test


In [4]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/visualize_gradcam.py" \
    --model_path "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/checkpoints/best_ayolo.pth" \
    --csv_path "/content/drive/MyDrive/Spring Semester/dataset/processed_metadata/rsna_master_metadata.csv" \
    --img_dir "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/visual_proofs" \
    --num_samples 15

🖼️  Generating explanations for 15 samples...
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:744: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
✅ Generated: 24e11e90-9a36-4ee9-8e27-6b58ba9ecab1 → NORMAL (1.4%)
✅ Generated: 481ff753-4106-405c-962e-a3f1c1e3bc37 → NORMAL (22.7%)
✅ Generated: f7a169e2-a1a7-439a-ab83-0d34cf5ae4b9 → NORMAL (4.2%)
✅ Generated: 6cdcd463-82a4-45b7-b780-ef366f9ecc62 → NORMAL (46.8%)
✅ Generated: 06a4f477-34ca-4625-945b-81bfa9e77273 → PNEUMONIA (64.5%)
✅ Generated: 813f2bf4-82b9-4b67-acdd-84aeb03a1149 → NORMAL (2.6%)
✅ Generated: e408ed59-e30a-4594-abbe-a4ec05f67e88 → NORMAL (33.7%)
✅ Generated: 9ed27b99-3235-476c-a752-abdc005fb46b → NORMAL (48.3%)
✅ Generated: b1c9ed5a-9be6-4f5e-a7f8-5eba7289c156 → 

In [ ]:
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/inference.py" \
    --model_path "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/checkpoints/best_ayolo.pth" \
    --img_path "/content/drive/MyDrive/Spring Semester/dataset/processed_metadata/rsna_master_metadata.csv" \
    --img_dir "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/inference_results" \
    --threshold 0.45 \
    --show_gradcam

### DEPLOYMENT

In [6]:
!pip install -q -U streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 8.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastapi 0.111.0 requires starlette<0.38.0,>=0.37.2, but you have starlette 1.0.0 which is incompatible.
mcp 1.27.0 requires httpx>=0.27.1, but you have httpx 0.27.0 which is incompatible.
mcp 1.27.0 requires uvicorn>=0.31.1; sys_platform != "emscripten", but you have uvicorn 0.30.1 which is incompatible.
gradio 5.50.0 requires fastapi<1.0,>=0.115.2, but you have fastapi 0.111.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.
gradio 5.50.0 requires starlette<1.0,>=0.40.0, but you have starlette 1.0.0 which is incompatible.
google-adk 1.29.0 requires fastapi<1.0.0,>=0.124.1, but you have fas

In [7]:
# ════════════════════════════════════════════════════════════════════
#  A-YOLO Deployment — 200 Epoch Model (run_final2)  |  Tek Hücre
# ════════════════════════════════════════════════════════════════════
import os, time, subprocess

ROOT = "/content/drive/MyDrive/Spring Semester/deep learning project"
DEPLOY = f"{ROOT}/src/a_yolo/a_yolo_deployment"
NGROK_TOKEN = "3D8UcQ5DZrvPgUdy1ph32b2ITL5_7Uxy1qkjdXwSUeS4WcDzJ"

# 1) Bağımlılıklar
print("📦 Bağımlılıklar yükleniyor...")
!pip install -q fastapi "uvicorn[standard]" python-multipart pyngrok nest-asyncio \
              streamlit requests grad-cam timm

# 2) Eski instance'ları öldür
print("🧹 Eski servisler temizleniyor...")
!pkill -f uvicorn 2>/dev/null
!pkill -f streamlit 2>/dev/null
from pyngrok import ngrok
ngrok.kill()
time.sleep(3)

# 3) Klasör yapısı + 200 epoch model dosyası
print("📁 Klasörler ve model kopyalanıyor...")
for sub in ["api/model", "api/utils", "ui"]:
    os.makedirs(f"{DEPLOY}/{sub}", exist_ok=True)
open(f"{DEPLOY}/api/model/__init__.py", "a").close()
open(f"{DEPLOY}/api/utils/__init__.py", "a").close()

!cp "{ROOT}/src/a_yolo/model.py"  "{DEPLOY}/api/model/ayolo.py"
!cp "{ROOT}/outputs/a_yolo/run_final2/checkpoints/best_ayolo.pth" \
    "{DEPLOY}/api/model/best_ayolo.pth"
print("   ✅ run_final2 (200 epoch) modeli kopyalandı")

# 4) FastAPI başlat
print("🚀 FastAPI başlatılıyor...")
api_proc = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=f"{DEPLOY}/api",
    stdout=open(f"{DEPLOY}/api.log", "w"),
    stderr=subprocess.STDOUT,
)
print("   ⏳ Model yükleniyor (~25 sn)...")
time.sleep(28)

# Sağlık kontrolü
health = subprocess.run(["curl", "-s", "http://localhost:8000/health"],
                        capture_output=True, text=True)
print(f"   API health: {health.stdout}")
if "ok" not in health.stdout:
    print("   ⚠️  API başlamadı. Log:")
    !tail -30 "{DEPLOY}/api.log"
    raise RuntimeError("API health check failed")

# 5) Streamlit başlat
print("🎨 Streamlit başlatılıyor...")
os.environ["API_URL"] = "http://localhost:8000/predict"
ui_proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501", "--server.address", "0.0.0.0",
     "--server.headless", "true", "--browser.gatherUsageStats", "false"],
    cwd=f"{DEPLOY}/ui",
    stdout=open(f"{DEPLOY}/ui.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(10)

# 6) ngrok tunnel
print("🌐 ngrok tunnel açılıyor...")
!ngrok config add-authtoken {NGROK_TOKEN}
public_url = ngrok.connect(8501, "http")

print("\n" + "═"*60)
print(f"✅ A-YOLO HAZIR — 200 Epoch Model (run_final2)")
print(f"   AUC=0.858 | mAP@50=0.586 | Recall=0.74")
print(f"\n🚀 URL: {public_url}")
print("═"*60)

📦 Bağımlılıklar yükleniyor...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 21.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mcp 1.27.0 requires httpx>=0.27.1, but you have httpx 0.27.0 which is incompatible.
mcp 1.27.0 requires uvicorn>=0.31.1; sys_platform != "emscripten", but you have uvicorn 0.30.1 which is incompatible.
gradio 5.50.0 requires fastapi<1.0,>=0.115.2, but you have fastapi 0.111.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.
gradio 5.50.0 requires starlette<1.0,>=0.40.0, but you have starlette 0.37.2 which is incompatible.
google-adk 1.29.0 requires fastapi<1.0.0,>=0.124.1, but you have fastapi 0.111.0 which is incompatible.
google-adk 1.29.0 requires PyYAML<7.0.0,>=6.0.2, but you have pyyaml 6.0.1 which is incompatible.
google-adk 1.29.0

In [9]:
# use_column_width → use_container_width
!sed -i 's/use_column_width=True/use_container_width=True/g' \
    "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/a_yolo_deployment/ui/app.py"

# Streamlit'i yeniden başlat
!pkill -f streamlit
import time, subprocess, os
time.sleep(3)

DEPLOY = "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/a_yolo_deployment"
os.environ["API_URL"] = "http://localhost:8000/predict"

ui_proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501", "--server.address", "0.0.0.0",
     "--server.headless", "true", "--browser.gatherUsageStats", "false"],
    cwd=f"{DEPLOY}/ui",
    stdout=open(f"{DEPLOY}/ui.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(8)
print("✅ Streamlit yeniden başladı")

✅ Streamlit yeniden başladı


### ABLATION

In [15]:
 # Cell A — Ablation runner (~6-9 saat)
!chmod +x "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/run_ablation.sh"
!bash "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/run_ablation.sh"


═══════════════════════════════════════════════════════
  ABLATION: mask=none  alpha=0.0  (20 epoch)
═══════════════════════════════════════════════════════
🚀  Device: cuda
⚠️  mask_strategy=none → SSL loss kapalı (alpha=0)
🎭  mask_strategy=none | ratio=0.75 | α=0.0
✅ 22681 kayıt işlendi (5110 bbox parse edildi)
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
✅ 4003 kayıt işlendi (902 bbox parse edildi)
📦  Train: 22681 | Val: 4003
Epoch 01 [Train]:   0% 0/709 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:143: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value

In [1]:
# Cell B — Tablo + grafik
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/aggregate_ablation.py"

✅ No-SSL (detection only): acc=0.804  prec=0.557  rec=0.626  f1=0.589  auc=0.825  map50=0.426  miou=0.367  iou50=0.351
✅ Random MAE + Detection: acc=0.832  prec=0.646  rec=0.567  f1=0.604  auc=0.853  map50=0.473  miou=0.382  iou50=0.346
✅ Gaussian Anatomy-Aware (Ours): acc=0.816  prec=0.617  rec=0.479  f1=0.539  auc=0.824  map50=0.439  miou=0.370  iou50=0.358

=== MARKDOWN TABLE ===
| Variant | Acc | Prec | Recall | F1 | AUC | mAP@50 | mIoU | IoU≥.5 |
|---|---|---|---|---|---|---|---|---|
| No-SSL (detection only) | 0.804 | 0.557 | 0.626 | 0.589 | 0.825 | 0.426 | 0.367 | 0.351 |
| Random MAE + Detection | 0.832 | 0.646 | 0.567 | 0.604 | 0.853 | 0.473 | 0.382 | 0.346 |
| Gaussian Anatomy-Aware (Ours) | 0.816 | 0.617 | 0.479 | 0.539 | 0.824 | 0.439 | 0.370 | 0.358 |

=== LATEX TABLE ===
\begin{tabular}{lcccccccc}
\toprule
Variant & Acc & Prec & Recall & F1 & AUC & mAP@50 & mIoU & IoU$\geq$.5 \\
\midrule
No-SSL (detection only) & 0.804 & 0.557 & 0.626 & 0.589 & 0.825 & 0.426 & 0.367 & 0.3

In [5]:
# Cell C — DÜZELTİLMİŞ Advanced Analysis (run_final2 modeli üzerinde)
!python "/content/drive/MyDrive/Spring Semester/deep learning project/src/a_yolo/advanced_analysis2.py" \
    --model_path "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/checkpoints/best_ayolo.pth" \
    --csv_path   "/content/drive/MyDrive/Spring Semester/dataset/processed_metadata/rsna_master_metadata.csv" \
    --img_dir    "/content/drive/MyDrive/Spring Semester/deep learning project/diffusion_guided_detr_data/images" \
    --output_dir "/content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/advanced_metrics" \
    --batch_size 64 \
    --mc_passes 20

✅ Model: /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/checkpoints/best_ayolo.pth
📂 test split'i seçildi. Satır sayısı: 2668
✅ 2668 kayıt işlendi (601 bbox parse edildi)
📊 2668 örnek üzerinde gelişmiş analiz başlıyor...
Analyzing: 100% 42/42 [17:57<00:00, 25.66s/it]

📈 ECE: 0.2033

══════════════════════════════════════════════════
        🚀 ERROR TAXONOMY
══════════════════════════════════════════════════
  True_Positive         : 242    (9.1%)
  Localization_Error    : 258    (9.7%)
  Background_Error      : 623    (23.4%)
  False_Negative        : 101    (3.8%)
  True_Negative         : 1444   (54.1%)
══════════════════════════════════════════════════

🎉 Sonuçlar: /content/drive/MyDrive/Spring Semester/deep learning project/outputs/a_yolo/run_final2/advanced_metrics


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  GitHub Release ZIP — A-YOLO Project
# ════════════════════════════════════════════════════════════════════
import os, shutil, zipfile, datetime
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Spring Semester/deep learning project")
STAGE = Path("/content/github_stage")
ZIP_OUT = Path(f"/content/a_yolo_github_{datetime.date.today()}.zip")

# Temizle
if STAGE.exists(): shutil.rmtree(STAGE)
STAGE.mkdir(parents=True)

# ── 1) Kaynak kod ──
shutil.copytree(ROOT / "src",      STAGE / "src",     dirs_exist_ok=True)
shutil.copytree(ROOT / "scripts",  STAGE / "scripts", dirs_exist_ok=True)
if (ROOT / "configs").exists():
    shutil.copytree(ROOT / "configs", STAGE / "configs", dirs_exist_ok=True)

# ── 2) Notebook ──
NB_DIR = STAGE / "notebooks"; NB_DIR.mkdir()
nb_candidates = list(ROOT.glob("*.ipynb"))
if nb_candidates:
    shutil.copy(nb_candidates[0], NB_DIR / "deep_learning_project.ipynb")

# ── 3) Asset görsel örnekleri (varsa eğitim eğrisi, gradcam) ──
ASSETS = STAGE / "assets"; ASSETS.mkdir()
for src, dst in [
    (ROOT / "outputs/a_yolo/run_final2/training_curves.png", "training_curves.png"),
    (ROOT / "outputs/a_yolo/run_final2/final_rsna_test/confusion_matrix.png", "confusion_matrix.png"),
    (ROOT / "outputs/a_yolo/run_final2/final_rsna_test/performance_curves.png", "roc_pr_curves.png"),
    (ROOT / "outputs/a_yolo/run_final2/advanced_metrics/calibration_curve.png", "calibration.png"),
    (ROOT / "outputs/a_yolo/run_final2/advanced_metrics/uncertainty_distribution.png", "uncertainty.png"),
]:
    if src.exists():
        shutil.copy(src, ASSETS / dst)

# Bir gradcam örneği
gradcam_dir = ROOT / "outputs/a_yolo/run_final2/visual_proofs"
if gradcam_dir.exists():
    samples = list(gradcam_dir.glob("*.png"))[:3]
    for i, s in enumerate(samples):
        shutil.copy(s, ASSETS / f"gradcam_example_{i+1}.png")

# ── 4) Otomatik dosya üret: requirements.txt ──
(STAGE / "requirements.txt").write_text("""\
torch>=2.0.0
torchvision>=0.15.0
timm>=0.9.0
albumentations>=1.4.18
opencv-python-headless>=4.8.0
numpy>=1.24.0
pandas>=2.0.0
scikit-learn>=1.3.0
matplotlib>=3.7.0
seaborn>=0.12.0
tqdm>=4.65.0
pydicom>=2.4.0
Pillow>=10.0.0
fastapi>=0.110.0
uvicorn[standard]>=0.27.0
streamlit>=1.32.0
python-multipart>=0.0.9
grad-cam>=1.5.0
pyngrok>=7.0.0
requests>=2.31.0
""")

# ── 5) .gitignore ──
(STAGE / ".gitignore").write_text("""\
# Python
__pycache__/
*.py[cod]
*$py.class
*.so
.Python
env/
venv/
.venv/
*.egg-info/
.ipynb_checkpoints/

# Model checkpoints (too large)
*.pth
*.pt
*.safetensors
outputs/
checkpoints/
runs/

# Datasets (too large)
data/
diffusion_guided_detr_data/
*.dcm
*.dicom

# Logs
*.log
api.log
ui.log

# OS
.DS_Store
Thumbs.db

# IDE
.vscode/
.idea/
*.swp

# Streamlit
.streamlit/secrets.toml

# Misc
.env
.cache/
""")

# ── 6) MIT LICENSE ──
(STAGE / "LICENSE").write_text(f"""\
MIT License

Copyright (c) {datetime.date.today().year} Cem [Soyadı]

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND.
""")

# ── 7) ZIP'le ──
shutil.make_archive(str(ZIP_OUT.with_suffix("")), "zip", STAGE)

# Boyut + indirme
size_mb = ZIP_OUT.stat().st_size / 1024 / 1024
print(f"✅ Hazır: {ZIP_OUT}")
print(f"📦 Boyut: {size_mb:.2f} MB")

from google.colab import files
files.download(str(ZIP_OUT))